In [7]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import aneurysm_3Dsegmentation_in_CTA.model_utility.tuning as tuner
import segmentation_models_3D as sm3
import keras_tuner as kt
import os

In [ ]:
'''
# run one time ...
src = "data/CTA nii"
pname = os.listdir(src)
cnt = 0
for name in pname :
    res = tts.merge(os.path.join(src , name))
    cnt+=1
else :
    print("done !!!")
'''

In [8]:
# loading data tensors
dataSource = "data/CTA nii" 

sizeContainer = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        sizeContainer["S"] ,
        sizeContainer["M"] ,
        sizeContainer["L"]
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet , mergeTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet  , mergeTestSet  = tts.dataTensorLoading(testSet)


In [10]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet , mergeTrainSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet , mergeTestSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")

data/CTA nii/AHMU1218096/cta_images_head_AHMU1218096.nii.gz
data/CTA nii/AHMU1218096/3D_aneurysm_AHMU1218096_@32_@MCA_@BERRY-label.nii
NoMerge
data/CTA nii/990784-TAHEREHM...BAZAN/990784-TAHEREHM...BAZAN_Brain_-CTA_20171010132707_4.nii
data/CTA nii/990784-TAHEREHM...BAZAN/990784-TAHEREHM...BAZAN_Brain_-CTA_20171010132707_4_@29_@BASILAR_@FUSIFORM-label.nii
NoMerge
data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4.nii
data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4_@37_@MCA_@BERRY-label.nii
NoMerge
data/CTA nii/1121445-AMENEKHAVARI/1121445-AMENEKHAVARI_CTA_20211120080858_4.nii
data/CTA nii/1121445-AMENEKHAVARI/1121445-AMENEKHAVARI_CTA_20211120080858_4_@16_@ACOM_@BLISTER-label.nii
NoMerge
data/CTA nii/1017563-KHADIJE...JANDI/1017563-KHADIJE...JANDI_Brain_-CTA_20180703064010_4.nii
data/CTA nii/1017563-KHADIJE...JANDI/1017563-KHADIJE...JANDI_Brain_-CTA_20180703064010_4_@46_@DACA_@BERRY-label.nii
NoMerge
data/CTA nii/AHMU121

In [11]:
# pipeline configuring

geo      = utl.randomGeo(p=0.5)
crop     = utl.volume_crop((128 , 128 , 128) , paddDim=80)
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 128 , 128 , 128 , 3] ,
    labelShape=[None , 128 , 128 , 128 , 1]
)

windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=0.5 ,
    p_ww=0.5
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64 , tf.float64])
def rimg(imgPath , labelPath , mergePath) :
    return utl.read_img(imgPath , labelPath , mergePath)
def read_img(img , label , merge) :
    imglbl = rimg(img , label , merge)
    img   = imglbl[0]
    label = imglbl[1]
    merge = imglbl[2]
    return img , label , merge

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label
# normalize by fix value for the raw values (no windowing)
normalizer = utl.fixedRangeNormalization(min_val=0 , max_val=3071 , clip=True)


In [12]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet , mergeTrainSet))
    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )

    .cache("myCacheTrain")
    .shuffle(buffer_size=20 , seed=42 , reshuffle_each_iteration=True)
    
    .map(
        rot ,
        num_parallel_calls=2
    )
    .map(
        geo.flipX ,
        num_parallel_calls=2
    )
    .map(
        geo.flipY ,
        num_parallel_calls=2
    )
    .map(
        geo.flipZ ,
        num_parallel_calls=2
    )
    .map(
        utl.channelize ,
        num_parallel_calls=4
    )
    .batch(batch_size=4)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        normalizer.apply ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTestSet , labelTestSet , mergeTestSet))

    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )
    .map(
        utl.channelize ,
        num_parallel_calls=4
    )
    .cache("myCacheValid")
    .batch(batch_size=2)
    
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        normalizer.apply ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
for data in dataloaderTrain.take(5) :
    print("image shape :" , data[0].shape)
    print("label shape :" , data[1].shape)

technical run

In [ ]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4_res18
)


# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze_Conv3_unit1ToEnd_res18 , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)


# compilation
lr = keras.optimizers.schedules.PiecewiseConstantDecay(
    [
        5530 ,
        11060 ,

    ] ,
    [
        1e-3 ,
        1e-4 ,
        1e-5
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice
    ] ,
)

logger = keras.callbacks.CSVLogger("tuning_logs/technical_seed.csv")

# model training
history = model.fit(
    x = dataloaderTrain ,
    epochs=250 ,
    validation_data = dataloaderValid ,
)

In [ ]:
print(model.summary())

Unet HyperModel

In [13]:
# building hyperModel
hp = kt.HyperParameters()


res18HyperModel = tuner.HyperUnetResNet(
    backbone="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features="default" ,
    steps=[
        5530 ,
        11060
    ]
)

res18HyperModel.set_tuning_param(
    unfreeze_point = [
        conf.unfreeze_Conv2_unit1ToEnd_res18 ,
        conf.unfreeze_Conv3_unit1ToEnd_res18 ,
        conf.unfreeze_Conv4_unit1ToEnd_res18  
    ] ,
    alphForFocalLoss=[0.25 , 0.35 , 0.45] ,
    alphaForWeightedFocalDiceloss=[
        [1.3] , 
        [0.2] , 
    ] ,

    learning_rate=[   
        [1e-3 , 5e-4] ,
        [1e-4] ,
        [1e-5]
    ]

)

In [ ]:
# tuning model
tuner = kt.GridSearch(
    res18HyperModel ,
    objective="val_loss" ,
    directory="tuning_logs" ,
    project_name="hypermodel_D1"
)

tuner.search(
    x = dataloaderTrain ,
    epochs=200 ,
    validation_data = dataloaderValid ,
)


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
stage2_unit1_conv1|stage2_unit1_conv1|unfreeze_point
0.25              |0.25              |alpha1
1.3               |1.3               |alpha2_1
0.2               |0.2               |alpha2_2
0.001             |0.001             |step 1
0.0001            |0.0001            |step 2
1e-05             |1e-05             |step 3

Epoch 1/200


/home/saadatia/Desktop/myVenv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1788024692.587102   53153 service.cc:153] XLA service 0x7b5cd887cdd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788024692.587145   53153 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1788024693.139882   53153 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788024696.626752   53153 cuda_dnn.cc:461] Loaded cuDNN version 92300
E0000 00:00:1788024705.065631   53153 cuda_timer.cc:87] Delay ke

78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - dice: 3.2051e-06 - loss: 0.2122 - v__recall: 1.9003

E0000 00:00:1788024892.041638   53154 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788024897.385025   53154 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788024915.622860   53153 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788024920.218100   53153 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


78/78 ━━━━━━━━━━━━━━━━━━━━ 257s 2s/step - dice: 3.2051e-06 - loss: 0.2122 - v__recall: 1.9003 - val_dice: 0.0000e+00 - val_loss: 0.2040 - val_v__recall: 0.0000e+00
Epoch 2/200
78/78 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - dice: 0.1459 - loss: 0.1961 - v__recall: 15.0990 - val_dice: 0.0000e+00 - val_loss: 0.2013 - val_v__recall: 0.0000e+00
Epoch 3/200
78/78 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - dice: 0.3940 - loss: 0.1597 - v__recall: 58.1639 - val_dice: 0.0000e+00 - val_loss: 0.2074 - val_v__recall: 0.0000e+00
Epoch 4/200
51/78 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - dice: 0.4467 - loss: 0.1160 - v__recall: 54.0173

In [ ]:
best_confguration = tuner.get_best_hyperparameters()[0].get_config()
pprint(best_confguration["values"])